In [ ]:
!pip install bertopic sentence-transformers umap-learn hdbscan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.1 MB/s eta 0:00:00


In [ ]:
# STEP 1: LOAD DATA AND PREPARE DOCUMENTS
# We use the processed_text column — lemmatised, stop words removed —
# as input for BERTopic. This reduces vocabulary noise and helps the
# clustering algorithm find meaningful semantic groupings.
# Any rows with missing processed_text are dropped before modeling.
import pandas as pd

df = pd.read_csv("youtube_comments_sentiment.csv")

df.head()

,author,updated_at,like_count,text,video_id,public,clean_text,tokens,lemmas,processed_text,vader_sentiment,vader_score,roberta_sentiment,roberta_score
0,@doomkid1331,2026-05-23T23:42:49Z,0,"Wore my Mk4s until they broke, and exchanged t...",XbXXKhfvSF0,True,wore my mk4s until they broke and exchanged th...,"['wore', 'my', 'mk4s', 'until', 'they', 'broke...","['wear', 'mk4s', 'break', 'exchange', 'mk5', '...",wear mk4s break exchange mk5 perfect mk6,Positive,0.1548,positive,0.455222
1,@TurkeyTomDeezNuts,2026-05-23T16:48:27Z,0,These have such bad reviews,XbXXKhfvSF0,True,these have such bad reviews,"['these', 'have', 'such', 'bad', 'reviews']","['bad', 'review']",bad review,Negative,-0.5423,negative,0.926344
2,@elevate5136,2026-05-22T04:30:00Z,0,I bought these in South Africa for my 12 hour ...,XbXXKhfvSF0,True,i bought these in south africa for my 12 hour ...,"['i', 'bought', 'these', 'in', 'south', 'afric...","['buy', 'south', 'africa', '12', 'hour', 'flig...",buy south africa 12 hour flight home absolutel...,Positive,0.6012,positive,0.949329
3,@Tdizzle_91,2026-05-21T20:57:05Z,0,Can you compare these to the new collextion mo...,XbXXKhfvSF0,True,can you compare these to the new collextion mo...,"['can', 'you', 'compare', 'these', 'to', 'the'...","['compare', 'new', 'collextion', 'model', 'kno...",compare new collextion model know buy lol,Positive,0.4215,neutral,0.837075
4,@andrewbarf,2026-05-21T09:35:58Z,0,Excellent review Marques; all the main points ...,XbXXKhfvSF0,True,excellent review marques all the main points c...,"['excellent', 'review', 'marques', 'all', 'the...","['excellent', 'review', 'marque', 'main', 'poi...",excellent review marque main point cover waffl...,Positive,0.7085,positive,0.958464


In [ ]:
df = df.dropna(subset=["processed_text"])

docs = df["processed_text"].astype(str).tolist()

print(len(docs))

24590


In [ ]:
# STEP 2: FIT BERTOPIC MODEL
# BERTopic is initialised with language="english" and calculate_probabilities=True.
# We do not set a fixed number of topics — BERTopic determines this automatically
# through HDBSCAN density-based clustering.
# This step takes several minutes. GPU speeds it up significantly.
from bertopic import BERTopic

topic_model = BERTopic(
    language="english",
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

2026-06-08 15:51:15,397 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/769 [00:00<?, ?it/s]

2026-06-08 15:55:16,857 - BERTopic - Embedding - Completed ✓
2026-06-08 15:55:16,858 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-08 15:56:11,379 - BERTopic - Dimensionality - Completed ✓
2026-06-08 15:56:11,381 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-08 15:59:21,325 - BERTopic - Cluster - Completed ✓
2026-06-08 15:59:21,342 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-08 15:59:21,876 - BERTopic - Representation - Completed ✓


In [ ]:
# STEP 3: INSPECT TOPIC RESULTS
# get_topic_info() shows all identified topics and their sizes.
# Topic -1 is the outlier class — comments too short or ambiguous to cluster.
# A high outlier count is expected for YouTube comments and is not a model failure.
# get_topic(n) shows the top keywords defining each topic.
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,8294,-1_watch_headphone_apple_like,"[watch, headphone, apple, like, battery, garmi...",[good design go oppo good camera go vivo video...
1,0,235,0_xiaomi_14_15_17,"[xiaomi, 14, 15, 17, 13, mi, xiaomis, max, 17p...","[xiaomi, xiaomi, xiaomi]"
2,1,209,1_chinese_china_brand_company,"[chinese, china, brand, company, north, wester...","[chinese not buy, chinese chinese chinese, w c..."
3,2,192,2_headphone_450_jack_premium,"[headphone, 450, jack, premium, audiophile, bu...","[well apple headphone, headphone, headphone]"
4,3,182,3_samsung_win_galaxy_hater,"[samsung, win, galaxy, hater, king, sms, lover...",[samsung samsung samsung samsung samsung samsu...
...,...,...,...,...,...
406,405,10,405_forerunner_965_5ks_consecutive,"[forerunner, 965, 5ks, consecutive, 4045, runn...",[use apple watch daily driver happy train swit...
407,406,10,406_wh1000xm4_incredible_wh1000xm5_wh1000xm6,"[wh1000xm4, incredible, wh1000xm5, wh1000xm6, ...",[wh1000xm5 pure garbage give mic issue call ex...
408,407,10,407_dawgand_x200_hideous_stunning,"[dawgand, x200, hideous, stunning, swap, diffi...","[x200 pro great stunning price good, x200 pro ..."
409,408,10,408_gaming_yearsmy_bezelless_inscreen,"[gaming, yearsmy, bezelless, inscreen, cameram...","[good gaming phone good game camera, talk came..."


In [ ]:
topic_model.get_topic(0)

[('xiaomi', np.float64(0.10008144004036132)),
 ('14', np.float64(0.0541178034213963)),
 ('15', np.float64(0.026724750761794622)),
 ('17', np.float64(0.019767777174858385)),
 ('13', np.float64(0.009128117663020139)),
 ('mi', np.float64(0.008947929583628193)),
 ('xiaomis', np.float64(0.00810184637602258)),
 ('max', np.float64(0.007938449312875448)),
 ('17pro', np.float64(0.007580076235966657)),
 ('contender', np.float64(0.00724478767194189))]

In [ ]:
topic_model.get_topic(1)

[('chinese', np.float64(0.08558519550418682)),
 ('china', np.float64(0.0524641030005162)),
 ('brand', np.float64(0.020020848787169542)),
 ('company', np.float64(0.01263927557519448)),
 ('north', np.float64(0.012507273839973264)),
 ('western', np.float64(0.009150065453607997)),
 ('market', np.float64(0.009142967256255278)),
 ('asian', np.float64(0.008587179014673242)),
 ('korean', np.float64(0.00829874255838596)),
 ('country', np.float64(0.008235009077880117))]

In [ ]:
# STEP 4: VISUALISE TOPICS
# visualize_barchart shows the top keywords for the top N topics.
# This is the main visual for the report — take a screenshot of this output.
# visualize_topics shows a 2D scatter plot of topic clusters.
topic_model.visualize_barchart(top_n_topics=10)

In [ ]:
topic_model.visualize_barchart(top_n_topics=10)
topic_model.visualize_topics()

In [ ]:
df.to_csv("youtube_comments_BERTtopics.csv", index=False)

In [ ]:
df["topic"] = topics

In [ ]:
from google.colab import files

files.download("youtube_comments_BERTtopics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>